The ``TreeModel`` object can be instantiated without providing any parameter:

In [1]:
import aaanalysis as aa
tm = aa.TreeModel()

You can provide a list of tree-based models and their respective arguments using the ``list_model_classes`` and ``list_model_kwargs`` parameters:

In [2]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier

# Classes used as default
list_model_classes = [RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier]
print("Default model arguments: ", tm._list_model_kwargs)

# Adjust default parameters
list_model_kwargs = [dict(n_estimators=64)] * 3
tm = aa.TreeModel(list_model_classes=list_model_classes, list_model_kwargs=list_model_kwargs)
print("New model arguments: ", tm._list_model_kwargs)

Default model arguments:  [{'random_state': None}, {'random_state': None}]
New model arguments:  [{'n_estimators': 64, 'random_state': None}, {'n_estimators': 64, 'random_state': None}, {'n_estimators': 64, 'random_state': None}]


You can set the ``random_state`` and ``verbose`` parameters: 

In [3]:
# Set random sed and disable verbosity
tm = aa.TreeModel(random_state=42, verbose=False)
print("New model arguments: ", tm._list_model_kwargs)

New model arguments:  [{'random_state': 42}, {'random_state': 42}]


You compare different feature pre-filtering strategies by utilizing the ``is_preselected`` parameter, which we will demonstrate using the ``DOM_GSEC`` example dataset and its respective feature set (see [Breimann25]_):

In [4]:
import numpy as np
aa.options["verbose"] = False # Disable verbosity

df_seq = aa.load_dataset(name="DOM_GSEC")
labels = df_seq["label"].to_list()
df_feat = aa.load_features(name="DOM_GSEC").head(100)

# Create feature matrix
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
X = sf.feature_matrix(features=df_feat["feature"], df_parts=df_parts)

# Pre-select top 10 and top 50 features
mask_top10 = np.asarray(df_feat.index < 10)
mask_top50 = np.asarray(df_feat.index < 50)

We can now compare the prediction performance for these preselected feature sets using the ``TreeModel().eval()`` method:

In [5]:
df_eval = tm.eval(X, labels=labels, list_is_selected=[np.array([mask_top10]), np.array([mask_top50])])
aa.display_df(df_eval)

,name,accuracy,precision,recall,f1
1,Set 1,0.758200,0.767700,0.761500,0.756700
2,Set 2,0.842200,0.838600,0.875000,0.849000


**Further parameters.** The constructor's ``is_preselected`` restricts every fitting round to a pre-filtered feature subset, letting you compare pre-filtering strategies against the full feature set.

In [6]:
# is_preselected restricts every fitting round to a pre-filtered feature subset (here the top 10)
tm = aa.TreeModel(is_preselected=mask_top10)
is_selected_top10 = tm.fit(X=X, labels=labels).is_selected_
print("Features selected within the top-10 pre-selection:", int(np.asarray(is_selected_top10).sum()))

Features selected within the top-10 pre-selection: 50
